# 30 — Claims, conclusions and paper outline: Wu 2003 recycle plant (§7-§9)

Synthesises all results from notebooks 20-29b and 31 into validated claims, a quantitative
dashboard, and a paper-section outline — with inline supporting figures. Styled after
`14_claims_and_conclusions.ipynb` (the propylene-oxide half of this same C&ChE paper);
this notebook covers the **Wu 2003 CSTR-column-recycle plant** half (article §7-§9).

> Every claim below is backed by a specific notebook and figure produced in that notebook.
> Where a session finding **retracted** an earlier claim, both the original claim and the
> retraction are shown — this paper's headline for this system is the retraction itself
> (§7.4/§8.1), not a hidden revision.

**Read `HANDOFF.md` before trusting any number in this notebook that isn't re-derived
live below** — this project went through two rounds of headline retraction on this system
(2026-07-03, 2026-07-05) and prose elsewhere (older notebook markdown, early article
drafts) may still contain retracted claims.

## 1. Setup

In [1]:
import pathlib
import pickle
import json

import numpy as np
import pandas as pd
from IPython.display import display, Image, HTML

ROOT    = pathlib.Path.cwd().parent
FIGS    = ROOT / 'figures'
RESULTS = ROOT / 'results'
SBI_LOGS = ROOT / 'sbi-logs'


def fig(fname, caption='', width='98%'):
    path = FIGS / fname
    if not path.exists():
        print(f'MISSING: {fname}')
        return
    display(HTML(
        f'<figure style="margin:4px 0"><img src="{path.as_posix()}" style="width:{width}"/>'
        f'<figcaption style="font-size:0.85em;color:#444">{caption}</figcaption></figure>'
    ))


def figs(*items, cols=2):
    cell_w = f'{98//cols}%'
    row = '<div style="display:flex; gap:1%">'
    for fname, caption in items:
        path = FIGS / fname
        if not path.exists():
            row += f'<div style="width:{cell_w}">MISSING: {fname}</div>'
            continue
        row += (f'<figure style="width:{cell_w}; margin:4px 0">'
                f'<img src="{path.as_posix()}" style="width:100%"/>'
                f'<figcaption style="font-size:0.85em;color:#444">{caption}</figcaption></figure>')
    row += '</div>'
    display(HTML(row))

print("Setup OK")

Setup OK


## 2. System and control-structure overview (nb20-nb22)

The Wu (2003) plant is a CSTR-column-recycle benchmark: a reactor feeds a distillation
column, and the column's bottoms/recycle streams close a snowball-effect loop back into
the reactor feed. Two control structures are compared throughout: **S-B** (base case, no
online composition analyser — 9 observed channels, 66-D summary statistics) and **S-A**
(adds a distillate composition analyser x_D — 10 channels, 72-D). Five degradation
parameters are inferred: **α** (catalyst activity), **β_r** (reactor jacket fouling),
**η_col** (column tray efficiency), **ξ_reb** (reboiler fouling), **z_A0_eff** (feed
purity).

In [2]:
figs(
    ('nb21_cl_vs_ol_masking.png', 'Fig — Closed-loop vs open-loop masking: Loop 1 zeros the reactor-temperature response to alpha/beta_r'),
    ('nb22_fr_norm_check.png',    'Fig — Recycle-flow snowball sanity check across the fault grid'),
    cols=2)

## 3. Identifiability structure — Fisher information (nb23)

Local FIM at the nominal operating point, computed via finite differences on the 66-D
S-B summary statistics (`compute_summaries`) — this is the paper's own headline
methodology for identifiability claims, later re-applied to a raw-trajectory
representation in §5 below when this section's own conclusions were re-examined.

**7.2.1 — T_r masking, confirmed shared with the propylene-oxide system.**
`∂T_r_ss/∂α ≡ ∂T_r_ss/∂β_r ≡ 0` under Loop 1 PI control (T_r held at setpoint):
T_r-derived features contribute **0.00%** to both I_αα and I_β_r. This is the *same*
scalar masking mechanism as the PO system's β (§6.3, nb15) — see §8.1 for the
cross-system comparison.

**7.2.2 — (α, β_r) FIM structure is qualitatively different from PO.** Because Wu 2003
has no observable concentration channel (z_A is an internal state, not in S-B), α and
β_r both excite the same physics correlations (`corr_Qreb_FR`, `corr_Qj_FR`,
`corr_Rn_Vn`) instead of each having its own channel:

In [3]:
fim_table = pd.DataFrame([
    {"Quantity": "I_alpha / I_beta_r",           "PO CSTR": "250-500x", "Wu 2003": "1.1-1.4x (nearly equal)"},
    {"Quantity": "Primary alpha channel",         "PO CSTR": "C (concentration, 60%)", "Wu 2003": "corr_Qreb_FR (77%)"},
    {"Quantity": "Primary beta/beta_r channel",   "PO CSTR": "T_c, Q_c (decoupled from C)", "Wu 2003": "corr_Qreb_FR (49%) -- same channel as alpha"},
    {"Quantity": "(alpha, beta_r) normalised FIM off-diagonal", "PO CSTR": "small", "Wu 2003": "+0.901 at nominal (near-degenerate)"},
])
fim_table

,Quantity,PO CSTR,Wu 2003
0,I_alpha / I_beta_r,250-500x,1.1-1.4x (nearly equal)
1,Primary alpha channel,"C (concentration, 60%)",corr_Qreb_FR (77%)
2,Primary beta/beta_r channel,"T_c, Q_c (decoupled from C)",corr_Qreb_FR (49%) -- same channel as alpha
3,"(alpha, beta_r) normalised FIM off-diagonal",small,+0.901 at nominal (near-degenerate)


In [4]:
fig('nb23_fim_heatmap.png',
    'Fig 8 -- 5x5 normalised FIM heatmaps, S-B (left) and S-A (right). Key features: beta_r and '
    'alpha both near-zero T_r contribution; high (alpha, beta_r) off-diagonal (+0.901) under the '
    '66-D summary-statistic representation -- shown in Sec. 5 to collapse to ~0.00 under the raw '
    'trajectory; eta_col off-diagonal near-zero at nominal in both structures.', width='70%')

**7.2.3 — (α, η_col) does not survive a full-feature-set check.** A *restricted*
identifiability scan using only recycle-flow (F_R) features shows a classic extended
banana ridge — the original headline. Combining F_R with the column's own T_reb-derived
features (both already standard S-B instrumentation) collapses this ridge to a localized
region. The calibrated S-B posterior corroborates: η_col 90% CI width ≈0.01-0.02
(essentially pinned), |corr(α, η_col)| < 0.15 at every tested point. **Retracted — see
§8.4 L4.**

**7.2.4 — z_A0 is the most locally identifiable parameter** (largest FIM diagonal,
I_z_A0 = 2.15e14 vs I_αα = 2.22e13) — feed purity affects the whole reactor steady state
through inlet composition, a channel not shared with any other parameter. (§6 below shows
this local-information advantage does not straightforwardly translate into reliable
*single-window* fault detection — a distinct, practical finding.)

## 4. SBI training and calibration (nb24, nb25)

**S-B: resolved via an 8-seed ensemble.** SNPE-C (`zuko_nsf`, 60 hidden units/3
transforms, trained on 15,000 simulations) is seed-unstable at this problem's
dimensionality (§8.4 L9) — identical architecture and data, different seed, can flip
an individual parameter's SBC from p≈0.6 to p≈1e-7. The adopted mitigation: train 8
seeds, corroborate each at multiple SBC sample sizes (N=200/400/800), and select by
worst-parameter KS p-value. **Seed 4 selected and used for every quantitative S-B claim
in this paper.**

In [5]:
with open(SBI_LOGS / 'wu2003_posterior_sb.pkl', 'rb') as f:
    sb_data = pickle.load(f)

print(f"Selected S-B posterior: seed={sb_data['selected_seed']}  "
      f"N_TRAIN={sb_data['N_TRAIN']}  arch={sb_data['arch']} "
      f"({sb_data['hidden_features']}/{sb_data['num_transforms']})")
print(f"Selection method: {sb_data['selection_method']}\n")

ens = pd.DataFrame(sb_data['ensemble_summary'])
ks = pd.json_normalize(ens['ks_pvalues']).add_prefix('ks_p_')
ens_table = pd.concat([ens[['seed', 'n_sbc', 'min_ks_pvalue']], ks], axis=1)
ens_table

Selected S-B posterior: seed=4  N_TRAIN=15000  arch=zuko_nsf (60/3)
Selection method: 8-seed ensemble, selected by max(min KS p-value across all 5 params) at N_SBC~400; seed 4 confirmed robust across 3 independent SBC draws at N=200/400/800 (see HANDOFF.md Finding 4/4b)



,seed,n_sbc,min_ks_pvalue,ks_p_alpha,ks_p_beta_r,ks_p_eta_col,ks_p_xi_reb,ks_p_z_A0_eff
0,0,400,8.385659e-08,0.613455,0.452936,8.385659e-08,4.097587e-03,0.914509
1,1,400,1.089165e-05,0.171282,0.452936,1.089165e-05,2.865285e-03,0.136559
2,2,400,4.985745e-02,0.107771,0.136559,2.612124e-01,4.985745e-02,0.107771
3,3,400,8.385659e-08,0.381461,0.697661,8.079292e-07,8.385659e-08,0.452936
4,4,400,1.365586e-01,0.317445,0.613455,1.365586e-01,4.529359e-01,0.171282
5,5,400,2.612124e-01,0.530890,0.452936,8.534115e-01,5.308901e-01,0.261212
6,6,400,2.126329e-01,0.779355,0.212633,3.174450e-01,9.585877e-01,0.452936
7,7,400,8.385659e-08,0.049857,0.530890,2.358199e-06,8.385659e-08,0.261212


**S-A: a settled negative result.** 0 of 40 total trained posteriors passed SBC at
N=400 across two sessions (16 seeds at the production architecture, plus 24 more testing
PCA feature reduction and both larger and smaller architectures — larger networks
performed *worse*, ruling out "network too small"). Root cause not identified; η_col and
ξ_reb fail most consistently, plausibly because S-A's composition-control loops actively
suppress the signal SNPE needs for these two parameters. **Do not use any S-A posterior
for a quantitative claim in this paper (§8.4 L10)** — the S-A posterior loaded in earlier
notebooks (seed 13) is retained only for the qualitative x_D-breaks-a-degeneracy argument
via simulator-only iso-contours, never for a trained-posterior number.

In [6]:
figs(
    ('nb24_marginal_posteriors_sb.png', 'Fig 7 (left) -- S-B marginal posteriors, seed-4 calibrated ensemble'),
    ('nb24_sbc_ranks_sb.png',           'Fig 7 (right) -- S-B SBC rank histograms, all 5 parameters pass'),
    cols=2)
figs(
    ('nb25_marginal_posteriors_sa.png', 'S-A marginal posteriors (uncalibrated -- shown as a limitation only, not a comparison)'),
    ('nb25_sbc_ranks_sa.png',           'S-A SBC rank histograms -- eta_col, xi_reb fail'),
    cols=2)

## 5. Headline: two apparent "banana" degeneracies, both artifacts (nb26, nb27, nb29b)

**This is the paper's central methodological finding for this system.** Two candidate
joint (2-parameter) degeneracies were investigated, in sequence, with progressively
stricter checks — and both failed to survive scrutiny, for two *different* reasons.

**Candidate 1 — (α, η_col) at W12: a restricted-*channel* artifact (§7.2.3 above).**
Retracted once F_R is combined with the column's own T_reb signal — both already
standard S-B instrumentation. No new sensor needed.

**Candidate 2 — (α, β_r) at W11: a lossy-*aggregation* artifact, one level deeper.**
Unlike Candidate 1, this pair survives combination of the *entire* 66-D summary-statistic
feature set — at the time, this looked like confirmation it was physical:
- Trained, calibrated seed-4 S-B posterior at W11: **correlation(α, β_r) = +0.998**.
- Executed EKF at W11 (`nb26`, tight tuning `P[6:,6:]≈1e-4`): **0% coverage** on both
  parameters.

Two independent follow-up checks, both using methodology this paper already establishes
elsewhere, overturn this reading:
1. **EKF tuning (nb27 §9).** The *same* EKF architecture with a more diffuse initial
   covariance (`P[6,6]=0.05, P[7,7]=0.02`) on the *identical* noisy W11 window converges
   to within ~0.3% of truth — reproducibly across 15+ noise seeds and 4 more grid points.
   The 0%-coverage result was a tuning artifact, not evidence of a genuine manifold.
2. **Raw-trajectory FIM (nb29b §4).** Re-deriving §3's own FIM methodology on the *raw,
   unaggregated* trajectory of the same 3 physical channels (T_r, T_j, F_R_norm) collapses
   the (α, β_r) normalised off-diagonal from ≈0.6-0.85 to **≈0.00**, reproducibly across
   noise seeds and at both the nominal point and W11. A negative control (a 108-D
   sub-window summary, 6x finer time resolution, same sensors) shows **no** improvement —
   ruling out an easy hand-crafted-feature fix. The lost information is in fine-grained
   transient *shape*, not coarser level/spread statistics.

In [7]:
figs(
    ('nb29b_alpha_etacol_retraction.png', 'Candidate 1 retraction: F_R-only ridge collapses once T_reb is combined'),
    ('nb29b_alpha_betar_confirmation.png','Candidate 2, Sec. 2 baseline: the (alpha, beta_r) ridge survives full-feature combination'),
    cols=2)
figs(
    ('nb29b_alpha_betar_richer_features.png', 'Sec. 3: finer time-resolution (108-D subwindow) does NOT narrow the ridge -- ambiguous on its own'),
    ('nb26_w11_headline.png',                 'nb26 W11 headline: SBI S-B banana vs. tightly-tuned EKF (0% coverage)'),
    cols=2)
fig('nb27_sequential_tracking.png',
    'nb27 Sec. 9: diffusely-tuned EKF (raw-trajectory access) tracks (alpha, beta_r) to ~1-2% at the '
    'same points nb26/nb29b Sec. 1-2 call non-identifiable -- the decisive corroboration.', width='90%')

**We report this as strong, convergent, two-independent-method evidence that
(α, β_r), like (α, η_col), is an artifact of the observation representation — but
explicitly do not claim to have fixed it.** Confirming this at the level of a trained,
calibrated SBI posterior on a raw-trajectory-aware representation (a CNN/RNN embedding
net, matching §6.3.3/`nb04b`'s methodology) is left to future work, given this system's
own documented SNPE training-instability at a *lower* feature dimensionality (§8.4 L9,
L10) — attempting and failing that retrain would be a worse outcome for this paper than
reporting the diagnostic finding on its own merits. **This decision was made explicitly
by the user this session; see `HANDOFF.md` for the full reasoning.**

**Design-guidance consequence.** The instrumentation recommendation this section would
otherwise have produced — "install a reactor-side concentration/heat-duty analyser" —
does not survive this reframing: the missing information was never absent from the
sensors, only from how their signal was compressed. **The higher-value intervention is
feature engineering (raw-trajectory-aware statistics or an embedding net), not new
sensors.**

## 6. η_col SBC investigation (nb29)

A parallel investigation, prior to nb29b, into why the pre-ensemble η_col posterior
failed SBC (p=0.0001). Diagnosis: the original 66-D feature set lacked a channel
specific to η_col rather than shared with α; the fix, `reb_per_boilup =
Q_reb / (V_norm * QREB_NOM)` (a column-heat-per-boilup-unit feature), together with the
8-seed ensemble/multi-N corroboration (§4 above), resolves the marginal calibration
failure. **Interpretive correction (per nb29b): a narrow η_col posterior at a tested
scenario is the *correct* answer (η_col is genuinely identifiable via T_reb), not a
residual approximation defect** — this reverses an earlier reading in nb29's own
markdown.

In [8]:
figs(
    ('nb29_two_sbc_failures.png',      'Original SBC failure: eta_col p=0.0001 (pre-fix, pre-ensemble)'),
    ('nb29_sensitivity_sweep.png',     'reb_per_boilup feature sensitivity sweep'),
    cols=2)

## 7. Fault classification (nb31)

Full detail in `31_wu2003_fault_classification.ipynb`. Posterior-mass classification
(same methodology as the PO system's nb11/§6.2), 14 closed-loop scenarios x 30 replicates,
calibrated seed-4 S-B posterior only (S-A excluded per §8.4 L10).

In [9]:
with open(RESULTS / '31_classification_summary.json') as f:
    cls_summary = json.load(f)

print(f"Overall accuracy: {cls_summary['accuracy']:.1%}   Macro-F1: {cls_summary['macro_f1']:.3f}")
print(f"({cls_summary['n_scenarios']} scenarios x {cls_summary['n_replicates_per_scenario']} replicates, "
      f"{cls_summary['n_posterior_samples_per_replicate']} posterior draws/replicate, "
      f"threshold={cls_summary['threshold']})\n")
print("Per-class F1:")
for c, f1 in cls_summary['per_class_f1'].items():
    print(f"  {c:10s} {f1:.3f}")

df_table10 = pd.read_csv(RESULTS / '31_fault_classification_metrics.csv')
df_table10

Overall accuracy: 87.4%   Macro-F1: 0.694
(14 scenarios x 30 replicates, 200 posterior draws/replicate, threshold=0.85)

Per-class F1:
  healthy    0.667
  reactor    0.948
  column     1.000
  feed       0.000
  multi      0.854


,scenario,true_unit,predicted_mode,scenario_accuracy,pooled_confidence,alpha_mean,beta_r_mean,eta_col_mean,xi_reb_mean,z_A0_eff_mean
0,W1_healthy,healthy,healthy,1.000,0.844,0.966,0.961,1.000,0.999,0.871
1,W2_cat_mild,reactor,reactor,1.000,0.660,0.752,0.841,1.000,1.001,0.812
2,W3_cat_severe,reactor,reactor,1.000,0.912,0.589,0.850,1.000,0.999,0.842
3,W4_cat_threshold,reactor,reactor,1.000,0.993,0.520,0.899,0.999,0.999,0.872
4,W5_jacket_mild,reactor,reactor,1.000,0.805,0.926,0.726,1.000,0.994,0.832
5,W6_jacket_severe,reactor,reactor,1.000,0.825,0.933,0.551,0.999,0.994,0.836
6,W7_col_eff_mild,column,column,1.000,0.734,0.937,0.916,0.801,1.002,0.845
7,W9_reb_fouling,column,column,1.000,0.825,0.963,0.957,0.999,0.699,0.869
8,W10_feed_impurity,feed,healthy,0.000,0.669,1.021,1.032,0.999,0.997,0.801
9,W11_reactor_combined,reactor,reactor,1.000,0.829,0.731,0.700,0.997,1.000,0.836


In [10]:
fig('nb31_confusion_matrix.png',
    'Table 10 source / confusion-matrix figure -- Wu 2003 S-B posterior-mass fault classification', width='55%')

**Key finding for §7.3/§8.1: the (α, β_r) representation artifact does not propagate
to unit-level fault classification.** W11 classifies as `reactor` in 30/30 replicates
despite the underlying (α, β_r) posterior correlation of 0.998 — because both parameters
map to the same fault unit, a smeared joint posterior still lands entirely inside the
correct region. The artifact corrupts *parameter-level attribution* within the reactor
unit, not *unit-level detection*. W12 (the retracted (α, η_col) candidate) also
classifies perfectly (30/30) as compound, with zero reactor/column leakage — confirming
§7.2.3's retraction at the classification level too.

**`RecycleScenarioConfig.fault_unit()` was corrected this session** — an earlier
name-substring-matching rule mislabelled W15 (`multi` via a "snowball" keyword despite only
α crossing threshold) and W13 (`reactor` via a "cat_" keyword despite two genuinely degraded
units). Rewritten to a pure parameter-threshold rule (matching the `cstr_sbi.luyben`
precedent); W15 now classifies correctly (30/30), and W13's corrected `multi` label exposes
a real weakness (7/30) rather than a spurious one.

**A third representation artifact was found and confirmed by FIM: (α/β_r, z_A0_eff).**
`nb31` §6b shows the feed-fault detection failure (F1 = 0.00, W10; F1 contribution from
W13) is **not** a single-window detection-power problem as originally suspected — it is a
genuine (α, z_A0_eff) and (β_r, z_A0_eff) near-degeneracy under `compute_summaries` (nominal
off-diagonal ≈ -0.12, jumping to ≈ -0.89 the moment α is degraded — the same magnitude as
the paper's own (α, β_r) headline number), which **collapses to ≈ -0.07 under the raw
trajectory** — the identical signature as the two already-documented artifacts. This
extends §8.1's "three for three" pattern: every joint near-degeneracy this paper has
surfaced via the 66-D summary-statistic representation and then checked against a
raw-trajectory FIM has turned out to be a representation artifact, not a physical property
of the plant.

## 8. Quantitative results dashboard

In [11]:
print('KEY NUMBERS FOR THE PAPER -- Wu 2003 recycle plant (article Sec. 7)')
print('=' * 70)
print(f"S-B posterior: seed {sb_data['selected_seed']}, N_TRAIN={sb_data['N_TRAIN']}, "
      f"{sb_data['arch']} {sb_data['hidden_features']}/{sb_data['num_transforms']}")
print(f"S-B ensemble selection: 8 seeds, multi-N (200/400/800) SBC corroboration")
print(f"S-A: 0/40 seeds passed SBC across two sessions -- settled negative (L10)")
print()
print("FIM (nb23, Sec. 3 above):")
print(f"  I_alpha / I_beta_r (Wu 2003):        1.1-1.4x   (PO CSTR: 250-500x)")
print(f"  (alpha, beta_r) normalised off-diag:  +0.901 at nominal (compute_summaries, 66-D)")
print(f"  (alpha, beta_r) normalised off-diag:  ~0.00     (raw 3-channel trajectory, nb29b Sec 4)")
print(f"  (alpha, eta_col) normalised off-diag: -0.142 at nominal (no persistent coupling)")
print(f"  I_z_A0 (most identifiable):          2.15e14  vs  I_alpha=2.22e13")
print()
print("W11 (alpha=0.80, beta_r=0.80) headline scenario:")
print(f"  SBI S-B posterior corr(alpha, beta_r):        +0.998")
print(f"  EKF (tight tuning, as originally deployed):   0% coverage both params (artifact)")
print(f"  EKF (diffuse tuning, identical data, nb27):   ~0.3% error, 15+ seeds robust")
print()
print(f"Fault classification (nb31, Sec. 7 above): "
      f"{cls_summary['accuracy']:.1%} accuracy, {cls_summary['macro_f1']:.3f} macro-F1")
for c, f1 in cls_summary['per_class_f1'].items():
    print(f"  {c:10s} F1={f1:.3f}")

KEY NUMBERS FOR THE PAPER -- Wu 2003 recycle plant (article Sec. 7)
S-B posterior: seed 4, N_TRAIN=15000, zuko_nsf 60/3
S-B ensemble selection: 8 seeds, multi-N (200/400/800) SBC corroboration
S-A: 0/40 seeds passed SBC across two sessions -- settled negative (L10)

FIM (nb23, Sec. 3 above):
  I_alpha / I_beta_r (Wu 2003):        1.1-1.4x   (PO CSTR: 250-500x)
  (alpha, beta_r) normalised off-diag:  +0.901 at nominal (compute_summaries, 66-D)
  (alpha, beta_r) normalised off-diag:  ~0.00     (raw 3-channel trajectory, nb29b Sec 4)
  (alpha, eta_col) normalised off-diag: -0.142 at nominal (no persistent coupling)
  I_z_A0 (most identifiable):          2.15e14  vs  I_alpha=2.22e13

W11 (alpha=0.80, beta_r=0.80) headline scenario:
  SBI S-B posterior corr(alpha, beta_r):        +0.998
  EKF (tight tuning, as originally deployed):   0% coverage both params (artifact)
  EKF (diffuse tuning, identical data, nb27):   ~0.3% error, 15+ seeds robust

Fault classification (nb31, Sec. 7 above): 87

## 9. Limitations specific to this system (article §8.4, L3/L4/L4'/L7/L9/L10)

In [12]:
limitations = pd.DataFrame([
    {"#": "L3",  "Limitation": "alpha bias ~0.10 downward under S-B",
     "Scope": "Wu 2003", "Mitigation": "Same structural mechanism as PO's beta bias (L2); reported in Sec. 4/7 above"},
    {"#": "L4",  "Limitation": "RETRACTED: (alpha, eta_col) is not a genuine joint non-identifiability",
     "Scope": "Wu 2003 S-B", "Mitigation": "Restricted-channel artifact; collapses once T_reb is combined with F_R (Sec. 3/5)"},
    {"#": "L4'", "Limitation": "RETRACTED: (alpha, beta_r) banana is very likely a summary-statistic aggregation artifact",
     "Scope": "Wu 2003, both structures", "Mitigation": "Lossy-aggregation artifact, one level deeper than L4; raw-trajectory FIM + diffuse-EKF corroboration (Sec. 5); fix (embedding net) identified, not implemented"},
    {"#": "L7",  "Limitation": "QSS column shortcut unstable for eta_col < 0.80",
     "Scope": "Wu 2003", "Mitigation": "W8, W14 removed from the scenario catalogue"},
    {"#": "L9",  "Limitation": "SNPE training is seed-unstable for the 5-param/66-72D posteriors",
     "Scope": "Wu 2003 S-A and S-B", "Mitigation": "8-seed ensemble + multi-N SBC corroboration (S-B only, Sec. 4)"},
    {"#": "L10", "Limitation": "S-A calibration is a settled, unresolved negative result",
     "Scope": "Wu 2003 S-A", "Mitigation": "0/40 seeds passed SBC across two sessions; do not use for any quantitative claim"},
    {"#": "L4''", "Limitation": "A THIRD representation artifact: (alpha/beta_r, z_A0_eff) near-degeneracy under compute_summaries, degrading feed/compound-fault classification (F1=0.00 for feed)",
     "Scope": "Wu 2003 S-B, fault classification (nb31 Sec. 6b)", "Mitigation": "FIM off-diagonal collapses from ~-0.89 (compute_summaries, alpha degraded) to ~-0.07 (raw trajectory) -- same signature as L4/L4'. NOT a single-window detection-power problem (an earlier diagnosis in this notebook/HANDOFF claiming pooling would help is retracted). Fix (embedding net) same as L4', not implemented."},
])
limitations

,#,Limitation,Scope,Mitigation
0,L3,alpha bias ~0.10 downward under S-B,Wu 2003,Same structural mechanism as PO's beta bias (L...
1,L4,"RETRACTED: (alpha, eta_col) is not a genuine j...",Wu 2003 S-B,Restricted-channel artifact; collapses once T_...
2,L4',"RETRACTED: (alpha, beta_r) banana is very like...","Wu 2003, both structures","Lossy-aggregation artifact, one level deeper t..."
3,L7,QSS column shortcut unstable for eta_col < 0.80,Wu 2003,"W8, W14 removed from the scenario catalogue"
4,L9,SNPE training is seed-unstable for the 5-param...,Wu 2003 S-A and S-B,8-seed ensemble + multi-N SBC corroboration (S...
5,L10,"S-A calibration is a settled, unresolved negat...",Wu 2003 S-A,0/40 seeds passed SBC across two sessions; do ...
6,L4'',A THIRD representation artifact: (alpha/beta_r...,"Wu 2003 S-B, fault classification (nb31 Sec. 6b)",FIM off-diagonal collapses from ~-0.89 (comput...


## 10. Pre-submission checklist (Wu 2003-specific items, article outline cross-check)

- [x] nb20-nb29b implementation and execution
- [x] S-B calibration resolved (8-seed ensemble + multi-N SBC)
- [x] S-A calibration exhaustively attempted, confirmed settled negative (L10)
- [x] (α, β_r)/(α, η_col) identifiability scans formalised (`nb29b`)
- [x] EKF run at W11, W12, W15 (`nb26`)
- [x] FIM analysis block (`nb23`, re-derived on raw trajectory in `nb29b` §4)
- [x] nb26 headline rewrite around the artifact-diagnostic framing
- [x] nb27 sequential tracking re-executed with the calibrated seed-4 posterior
- [x] nb24/nb25 assessment cells reflect the (α, β_r) finding
- [x] nb29 η_col SBC investigation follow-up note
- [x] **nb30 claims-and-conclusions synthesis (this notebook)**
- [x] **nb31 fault classification, framed as a worked example of the artifact diagnostic**
- [ ] All figures regenerated at publication quality (300 dpi, double-column)
- [ ] Table 10 (per-scenario classification) transcribed into the manuscript from
      `results/31_fault_classification_metrics.csv`
- [ ] Feed-fault detection limitation (new, §9 above) added to article §8.4 limitations table